### Imports and loads unique coordinates

In [1]:
import geopandas as gpd
import pandas as pd
import shapely
from pathlib import Path
from shapely.geometry import Point

In [2]:
PBF = "./data/osm/moscow.osm.pbf"

points = gpd.read_parquet("./data/clean.parquet")[["geo_lat", "geo_lon", "geometry"]]
points = points.drop_duplicates(subset=["geo_lat", "geo_lon"])
points = points.to_crs(32637)
points = points.reset_index(drop=True)
points.shape

(46639, 3)

In [3]:
def nearest_distance(points, layer):
    nearest = gpd.sjoin_nearest(points[["geometry"]], layer[["geometry"]], distance_col="distance")
    return nearest.groupby(level=0)["distance"].first()


def sum_in_buffer(points, layer, radius, value):
    buf = points.buffer(radius)
    pairs = gpd.sjoin(gpd.GeoDataFrame(geometry=buf), layer[["geometry"]], predicate="intersects")

    clipped = shapely.intersection(
        buf.loc[pairs.index].values,
        layer.geometry.loc[pairs["index_right"]].values,
    )

    if value == "length":
        pairs["value"] = shapely.length(clipped)
    else:
        pairs["value"] = shapely.area(clipped)

    if "weight" in layer.columns:
        pairs["value"] = pairs["value"] * layer["weight"].loc[pairs["index_right"]].values

    return pairs.groupby(level=0)["value"].sum().reindex(points.index, fill_value=0)

### Metro

In [4]:
metro = gpd.read_file(PBF, layer="points", where="""other_tags LIKE '%"station"=>"subway"%'""")
metro = metro.to_crs(32637)
metro["name"].nunique()

244

In [5]:
metro = metro.dissolve(by="name").reset_index()
metro["geometry"] = metro.geometry.centroid
metro.shape

(244, 11)

In [6]:
points["distance_metro"] = nearest_distance(points, metro)

### Rails

In [7]:
rail = gpd.read_file(
    PBF,
    layer="points",
    where="""(other_tags LIKE '%"railway"=>"station"%' OR other_tags LIKE '%"railway"=>"halt"%')
             AND other_tags NOT LIKE '%"station"=>"subway"%'""",
)
rail = rail.to_crs(32637)
rail["name"].nunique()

245

In [8]:
rail = rail.dissolve(by="name").reset_index()
rail["geometry"] = rail.geometry.centroid
rail.shape

(245, 11)

In [9]:
points["distance_rail"] = nearest_distance(points, rail)

### Roads

In [10]:
roads = gpd.read_file(
    PBF,
    layer="lines",
    where="highway IN ('motorway', 'trunk', 'primary', 'secondary', 'tertiary')",
)
roads = roads.to_crs(32637)
roads = roads.explode(index_parts=False).reset_index(drop=True)
roads.shape

(43251, 11)

In [11]:
roads["lanes"] = roads["other_tags"].str.extract(r'"lanes"=>"(\d+)')[0].astype(float)
roads["lanes"] = roads["lanes"].fillna(roads.groupby("highway")["lanes"].transform("median"))
roads["lanes"] = roads["lanes"].fillna(1)

In [12]:
highway_weights = {"motorway": 5, "trunk": 4, "primary": 3, "secondary": 2, "tertiary": 1}

roads["weight"] = roads["highway"].map(highway_weights) * roads["lanes"]

In [13]:
major_roads = roads[roads["highway"].isin(["motorway", "trunk"])]

points["distance_major_road"] = nearest_distance(points, major_roads)

In [14]:
points["road_load_300"] = sum_in_buffer(points, roads, 300, "length")
points["road_load_1000"] = sum_in_buffer(points, roads, 1000, "length")

In [15]:
center = [Point(37.6212, 55.7535)]
center = gpd.GeoDataFrame(
        {
        "id": [1],
        "name": ["center"],
        }, 
        geometry=center, 
        crs="EPSG:4326"
)
center = center.to_crs(32637)


In [16]:
points["dist_to_center"] = nearest_distance(points, center)


In [17]:
import numpy as np

center_point = center.geometry.iloc[0]

dx = points.geometry.x - center_point.x
dy = points.geometry.y - center_point.y

angle = np.arctan2(dx, dy)

points["azimuth_sin"] = np.sin(angle)
points["azimuth_cos"] = np.cos(angle)



### Save

In [18]:
features = pd.DataFrame(points.drop(columns="geometry"))

assert features.shape[0] == features[["geo_lat", "geo_lon"]].drop_duplicates().shape[0]
assert features.isna().sum().sum() == 0

Path("./data/geo").mkdir(exist_ok=True)
features.to_parquet("./data/geo/transport.parquet", index=False)